In [1]:
!pip install nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--

     ━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━ 0.5/1.5 MB 16.6 MB/s eta 0:00:01

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.8 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

import nltk
nltk.download('punkt_tab')


nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
import re
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.svm import SVC


[nltk_data] Downloading package punkt_tab to /root/nltk_data...


[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


[nltk_data] Downloading package stopwords to /root/nltk_data...


[nltk_data]   Unzipping corpora/stopwords.zip.


[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
file_path = '/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv'

# Read the CSV file into a pandas DataFrame
df = pd.read_csv(file_path)

# Display the first 5 rows to verify it loaded correctly
print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [4]:
def preprocess_text(text):
    clean_text = re.sub(r'<br\s*/>', ' ', text)
    clean_text = re.sub(r'<.*?>', ' ', clean_text)
    clean_text = clean_text.lower()
    clean_text = re.sub(r'[^a-zA-Z\s]', '', clean_text)
    
    tokens = nltk.word_tokenize(clean_text)
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [word for word in tokens if word not in stop_words]
    
    lemmatizer = WordNetLemmatizer()
    lemmatized_tokens = [lemmatizer.lemmatize(word) for word in filtered_tokens]
    
    return " ".join(lemmatized_tokens)


In [5]:
print("Starting text preprocessing...")
df['cleaned_review'] = df['review'].apply(preprocess_text)
print("Preprocessing complete.")

# Convert string labels to numerical labels
# 'positive' -> 1, 'negative' -> 0
df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})

# Get features (X) and labels (y)
X = df['cleaned_review']
y = df['sentiment']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Starting text preprocessing...


Preprocessing complete.


/tmp/ipykernel_74/673496461.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'positive': 1, 'negative': 0})


In [6]:
vectorizer = TfidfVectorizer(min_df=5, max_df=0.8)

# Fit and transform the training data
X_train_tfidf = vectorizer.fit_transform(X_train)

# Transform the test data using the same vectorizer
X_test_tfidf = vectorizer.transform(X_test)

# Initialize and train the Logistic Regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Make predictions and evaluate
y_pred = model.predict(X_test_tfidf)
print("\n--- Model Evaluation ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


--- Model Evaluation ---


Accuracy: 0.8957

Classification Report:


              precision    recall  f1-score   support

           0       0.91      0.88      0.89      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [7]:
svm_model = SVC(kernel='linear')

# Train the model on the TF-IDF features
print("Training the SVM model...")
svm_model.fit(X_train_tfidf, y_train)
print("Training complete.")

# Make predictions on the test data
print("Making predictions on the test set...")
y_pred_svm = svm_model.predict(X_test_tfidf)
print("Predictions complete.")

Training the SVM model...


Training complete.
Making predictions on the test set...


Predictions complete.


In [8]:
print("\n--- SVM Model Evaluation ---")
accuracy_svm = accuracy_score(y_test, y_pred_svm)
print(f"Accuracy: {accuracy_svm:.4f}")

# Print a detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_svm))


--- SVM Model Evaluation ---


Accuracy: 0.8973



Classification Report:


              precision    recall  f1-score   support

           0       0.90      0.89      0.90      4961
           1       0.89      0.91      0.90      5039

    accuracy                           0.90     10000
   macro avg       0.90      0.90      0.90     10000
weighted avg       0.90      0.90      0.90     10000



In [9]:
nb_model = MultinomialNB()

# Train the model on the TF-IDF features
print("Training the Naive Bayes model...")
nb_model.fit(X_train_tfidf, y_train)
print("Training complete.")

# Make predictions on the test data
print("Making predictions on the test set...")
y_pred_nb = nb_model.predict(X_test_tfidf)
print("Predictions complete.")

print("\n--- Naive Bayes Model Evaluation ---")
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"Accuracy: {accuracy_nb:.4f}")

# Print a detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_nb))

Training the Naive Bayes model...


Training complete.
Making predictions on the test set...


Predictions complete.

--- Naive Bayes Model Evaluation ---


Accuracy: 0.8661

Classification Report:


              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4961
           1       0.87      0.86      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

